In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1, MTCNN
from PIL import Image, ImageFilter, ImageEnhance
import numpy as np

d:\Repositories\uni-artificial-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
image_name_to_person_id: dict[str, int] = dict()
person_id_to_image_names: dict[int, list[str]] = dict()

with open("identity_CelebA.txt", "r") as lines:
    for line in lines:
        image_name, person_id_str = line.split(" ")
        image_name: str = image_name.strip()
        person_id: int = int(person_id_str.strip())
        image_name_to_person_id[image_name] = person_id
        
        if person_id not in person_id_to_image_names:
            person_id_to_image_names[person_id] = []
        person_id_to_image_names[person_id].append(image_name) 

In [ ]:
import os
from facenet_pytorch.models.inception_resnet_v1 import InceptionResnetV1
from torch import Tensor

DATA_DIR = "img_align_celeba/"

class FaceVerificationMLP(nn.Module):
    '''Define the MLP model that takes the difference between two embeddings'''
    def __init__(self, input_dim=512):
        super(FaceVerificationMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            # Output: [same_person_prob, different_person_prob]
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.model(x)


# Optional Preprocessing: Resize, smooth, convert to tensor, and normalize the image if they require it in one of the tasks. Note: this is a sample transoformation, modify it accoringly to the task
transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor()
])

# Load MTCNN for face detection, note: you should adjust the size wrt data and model, it can be done directly here or in the transform defined above
mtcnn = MTCNN(image_size=160, margin=20)

# Load FaceNet for embedding extraction (we'll be using a pretrained model)
facenet: InceptionResnetV1 = InceptionResnetV1(pretrained='vggface2').eval()


# Function to get face embedding from an image path
def get_embedding(image_path):
    img = Image.open(DATA_DIR + image_path).convert('RGB')
    # When needed add the preprocessing img = transform(img)
    face = mtcnn(img)  # returns a cropped, aligned face
    if face is None:
        raise ValueError(f"No face detected in {image_path}")
    face_embedding = facenet(face.unsqueeze(0))  # Add batch dimension
    return face_embedding.detach()



def get_diff_vector(img1_path, img2_path) -> Tensor:
    '''Function to compare two images and get the absolute difference vector'''
    
    emb1 = get_embedding(img1_path)
    emb2 = get_embedding(img2_path)
    return torch.abs(emb1 - emb2)


image_names: list[str] = os.listdir(DATA_DIR)

N = 10

ids_images: list[tuple[int, str]] = []

ids_images_desired_length = N * 2

train_data: set[tuple[str, str, int]] = set()

taken_indexes = set()
while len(ids_images) < ids_images_desired_length:
    while True:
        index = np.random.randint(0, len(image_names))
        if index not in taken_indexes:
            taken_indexes.add(index)
            break
    image_name = image_names[index]
    person_id = image_name_to_person_id[image_name]
    
    if len(person_id_to_image_names[person_id]) < 2:
        continue
    
    person_images: list[str] = person_id_to_image_names[person_id]
    
    needed = ids_images_desired_length - len(ids_images)
    if needed > 5:
        needed = 5
    ids_images.extend(list(map(lambda p_img: (person_id, p_img), person_images))[:needed])    
    
ids_used = set()    

while len(train_data) < N:
    person_id, img1_name = ids_images.pop(np.random.randint(0, len(ids_images)))
    img2_name = None
    img2_index = None
    img2_id = None
    
    # Ensure we have a different image for the second one
    while img2_name is None or img2_name == img1_name:
        img2_index = np.random.randint(0, len(ids_images))
        img2_name = ids_images[img2_index][1]
        img2_id = ids_images[img2_index][0]
    
    label = 1 if person_id == img2_id else 0
    
    ids_used.add(img2_id)
    ids_used.add(person_id)
    
    train_data.add((img1_name, img2_name, label))
    
print(ids_used)
    
# Prepare training tensors
X_train = []
y_train = []
for img1, img2, label in train_data:
    diff = get_diff_vector(img1, img2)
    X_train.append(diff.squeeze(0))
    y_train.append(label)

X_train = torch.stack(X_train)
y_train = torch.tensor(y_train)

# Define model, loss, optimizer (note: this is an example, adjust it according to the task)
model = FaceVerificationMLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the MLP model (note: this is an example, adjust the number of epochs and additional stop criteria to the task in the Assignement 5 list)
epochs = 20
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# Prediction Example, for additional experiments you may want to return the decision in numeric form or add model certenity


def predict_same_person(img1_path, img2_path, model):
    model.eval()
    diff = get_diff_vector(img1_path, img2_path)
    output = model(diff)
    _, predicted = torch.max(output, 1)
    return 'Same person' if predicted.item() == 1 else 'Different people'

# Example augmentation function


def augment_image(image, augment_type="gaussian_noise"):
    if augment_type == "gaussian_noise":
        # Add Gaussian noise
        image_np = np.array(image).astype(np.float32)
        # Modify the parameter value to adjust noise level if needed
        noise = np.random.normal(0, 25, image_np.shape)
        noisy_image = image_np + noise
        noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)
        augmented = Image.fromarray(noisy_image)
    elif augment_type == "blur":
        # Apply Gaussian blur with radius 3
        augmented = image.filter(ImageFilter.GaussianBlur(radius=3))
    elif augment_type == "increased_lighting":
        # Increase brightness by 50%
        enhancer = ImageEnhance.Brightness(image)
        augmented = enhancer.enhance(1.5)
    else:
        augmented = image
    return augmented


try:
    result = predict_same_person('000001.jpg', '000002.jpg', model)
    print("Prediction result:", result)
except ValueError as e:
    print("Error:", e)
